In [59]:
import gym
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

# 비트코인 거래 환경 정의
class TradingEnv(gym.Env):
    def __init__(self, data):
        super(TradingEnv, self).__init__()
        self.data = data
        self.current_step = 0
        self.action_space = gym.spaces.Discrete(3)  # 0: 매도, 1: 관망, 2: 매수
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(len(data.columns),), dtype=np.float32)
        self.balance = 10000  # 초기 자본
        self.holdings = 0

    def reset(self):
        self.current_step = 0
        self.balance = 10000
        self.holdings = 0
        return self.data.iloc[self.current_step].values

    def step(self, action):
        self.current_step += 1
        done = self.current_step >= len(self.data) - 1
        reward = 0

        price = self.data.iloc[self.current_step]['Close']
        if action == 2 and self.balance > 0:  # 매수
            self.holdings += self.balance/price
            self.balance -= self.holdings*price
            #print(f'balance:{self.balance}, holdings:{self.holdings}')
        elif action == 0 and self.holdings > 0:  # 매도
            self.balance += self.holdings*price
            reward += self.holdings*price  # 이익 반영
            self.holdings = 0
            #print(f'balance:{self.balance}, holdings:{self.holdings}, reward:{reward}')

        next_state = self.data.iloc[self.current_step].values
        return next_state, reward, done, {}

# IQN 네트워크 정의
class IQN(nn.Module):
    def __init__(self, state_dim, action_dim, quantiles=64):
        super(IQN, self).__init__()
        self.quantiles = quantiles
        self.action_dim = action_dim
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        self.quantile_fc = nn.Linear(128, action_dim * quantiles)
    
    def forward(self, state, taus):
        x = self.fc(state)
        quantile_values = self.quantile_fc(x)
        quantile_values = quantile_values.view(-1, self.quantiles, self.action_dim)
        return quantile_values

# IQN 에이전트 정의
class IQNAgent:
    def __init__(self, state_dim, action_dim, lr=0.001, gamma=0.99, quantiles=64):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.quantiles = quantiles
        self.memory = deque(maxlen=10000)
        self.batch_size = 64
        self.model = IQN(state_dim, action_dim, quantiles)
        self.target_model = IQN(state_dim, action_dim, quantiles)
        self.target_model.load_state_dict(self.model.state_dict())
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)

    def select_action(self, state, epsilon=0.1):
        if random.random() < epsilon:
            return random.randint(0, self.action_dim - 1)
        state = torch.FloatTensor(state).unsqueeze(0)
        taus = torch.rand(self.quantiles).unsqueeze(0)
        quantile_values = self.model(state, taus).mean(dim=1)
        return torch.argmax(quantile_values).item()
    
    def update(self):
        if len(self.memory) < self.batch_size:
            return
        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        states = torch.FloatTensor(states)
        actions = torch.LongTensor(actions).unsqueeze(-1)
        rewards = torch.FloatTensor(rewards).unsqueeze(-1)
        next_states = torch.FloatTensor(next_states)
        dones = torch.FloatTensor(dones).unsqueeze(-1)
        taus = torch.linspace(0, 1, self.quantiles).repeat(self.batch_size, 1)
        
        current_q_values = self.model(states, taus).gather(2, actions.unsqueeze(1).expand(-1, self.quantiles, -1))
        next_q_values = self.target_model(next_states, taus).mean(dim=1).max(dim=1, keepdim=True)[0]
        target_q_values = rewards + self.gamma * next_q_values * (1 - dones)
        loss = (torch.abs(current_q_values - target_q_values.unsqueeze(1))).mean()
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.target_model.load_state_dict(self.model.state_dict())

# 데이터 로드 및 환경 설정
data = pd.read_csv(f'/workspace/data/raw/BTCUSDT/BTCUSDT-1h-2022.csv', index_col=0)
data = data[['Open','High','Low','Close']]


env = TradingEnv(data)
agent = IQNAgent(state_dim=len(data.columns), action_dim=3)

# 학습 과정
episodes = 1
gamma = 0.99
for episode in range(episodes):
    state = env.reset()
    done = False
    total=0
    while not done:
        action = agent.select_action(state)
        
        next_state, reward, done, _ = env.step(action)
        agent.memory.append((state, action, reward, next_state, done))
        state = next_state
        total+=reward
    agent.update()
    print(f'episode:{episode}, total:{total}')

# 테스트 실행
state = env.reset()
done = False
totalreward=0
cnt=0
while not done:
    cnt+=1
    action = agent.select_action(state, epsilon=0)
    if action == 2 or action == 1:
        print(action)
    state, reward, done, _ = env.step(action)
    #print(reward)
    totalreward+=reward
    #print(f'cnt:{cnt}, state:{state}, reward:{reward}')
print(f'cnt:{cnt}, totalreward:{totalreward}')

episode:0, total:268085.2361085244
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
2
1
2
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
2
1
1
1
1
1
1
1
1
2
1
2
2
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
1
2
2
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
1
1
1
2
2